In [28]:
import os
import glob
import pandas as pd

RAW_DIR = os.path.join("data", "raw")
PROCEED_DIR = os.path.join("data", "proceed")
os.makedirs(PROCEED_DIR, exist_ok=True)

target_keywords = {
    "CLTUR_ARTS": "cltur",
    "PRSN": "prsn"
}

all_domain_dfs = []

In [29]:
# 1. 키워드별 CSV 추출 및 "data_manage_keyword" 추가
for keyword, tag in target_keywords.items():
    print("\n" + "=" * 60)
    print(f"[{keyword}] 1차 CSV 파일 수집 및 단독 중복 제거 시작")
    print("="*60)

    # 해당 키워드가 포함된 모든 CSV 파일 탐색
    csv_pattern = os.path.join(RAW_DIR, f"*{keyword}*.csv")
    csv_files = glob.glob(csv_pattern)
    
    print(f"발견된 CSV 파일 ({len(csv_files)}개):")
    csv_df_list = []
    
    for file in csv_files:
        try:
            temp_df = pd.read_csv(file, encoding="utf-8")
            temp_df.columns = temp_df.columns.str.lower()
            
            # 개별 파일별 행 개수 출력
            print(f" ➔ [{os.path.basename(file)}: {len(temp_df)}행")
            csv_df_list.append(temp_df)
        except Exception as e:
            print(f" ➔ [[파일 로드 실패] ({os.path.basename(file)}): {e}")
            
    if csv_df_list:
        df_domain_raw = pd.concat(csv_df_list, ignore_index=True)
        before_count = len(df_domain_raw)
        
        # 각 키워드 내에서 data_manage_no 중복 제거
        pk_col = "data_manage_no"
        if pk_col in df_domain_raw.columns:
            df_domain_cleaned = df_domain_raw.drop_duplicates(subset=[pk_col], keep="first")
            after_count = len(df_domain_cleaned)
            print(f" ➔ [{keyword}] 내부 중복 제거 전: {before_count}행 -> 제거 후: {after_count}행 (삭제: {before_count - after_count}행)")
        else:
            df_domain_cleaned = df_domain_raw
            print(f" ➔ [[경고] {pk_col} 컬럼이 없어 중복 제거를 건너뜁니다.")
        
        # 고유 구분을 위한 data_manage_keyword 컬럼 추가
        df_domain_cleaned["data_manage_keyword"] = tag
        print(f" ➔ 컬럼 추가 완료: data_manage_keyword = "{tag}"")
        
        # 보관용 파일 정렬
        df_domain_cleaned = df_domain_cleaned.sort_values(by=pk_col).reset_index(drop=True)
        
        # 키워드별 개별 마스터 파일 저장
        domain_output_name = f"kf_area_{tag}_merged.csv"
        domain_output_path = os.path.join(PROCEED_DIR, domain_output_name)
        df_domain_cleaned.to_csv(domain_output_path, index=False, encoding="utf-8-sig")
        print(f" ➔ [개별 마스터 저장 완료: {domain_output_path}")
        
        # 리스트에 축적
        all_domain_dfs.append(df_domain_cleaned)
    else:
        print(f" ➔ [{keyword}] 일치하는 CSV 데이터가 없습니다.")


[CLTUR_ARTS] 1차 CSV 파일 수집 및 단독 중복 제거 시작
발견된 CSV 파일 (7개):
 ➔ [KF_AREA_CLTUR_ARTS_DATA_LIST_202112.csv: 2071행
 ➔ [KF_AREA_CLTUR_ARTS_DATA_LIST_202106.csv: 2070행
 ➔ [KF_AREA_CLTUR_ARTS_DATA_LIST_201812.csv: 2070행
 ➔ [KF_AREA_CLTUR_ARTS_DATA_LIST_201806.csv: 2070행
 ➔ [KF_AREA_CLTUR_ARTS_DATA_LIST_202012.csv: 2070행
 ➔ [KF_AREA_CLTUR_ARTS_DATA_LIST_202006.csv: 2070행
 ➔ [KF_AREA_CLTUR_ARTS_DATA_LIST_201912.csv: 2070행
 ➔ [CLTUR_ARTS] 내부 중복 제거 전: 14491행 -> 제거 후: 14491행 (삭제: 0행)
 ➔ 컬럼 추가 완료: data_manage_keyword = 'cltur'
 ➔ [개별 마스터 저장 완료: data/proceed/kf_area_cltur_merged.csv

[PRSN] 1차 CSV 파일 수집 및 단독 중복 제거 시작
발견된 CSV 파일 (8개):
 ➔ [KF_AREA_PRSN_DATA_LIST_202106.csv: 1548행
 ➔ [KF_AREA_PRSN_DATA_LIST_202112.csv: 1549행
 ➔ [KF_AREA_PRSN_DATA_LIST_201806.csv: 1548행
 ➔ [KF_AREA_PRSN_DATA_LIST_201812.csv: 1548행
 ➔ [KF_AREA_PRSN_DATA_LIST_202006.csv: 1548행
 ➔ [KF_AREA_PRSN_DATA_LIST_202012.csv: 1548행
 ➔ [KF_AREA_PRSN_DATA_LIST_201906.csv: 1548행
 ➔ [KF_AREA_PRSN_DATA_LIST_201912.csv: 1548행
 ➔ [PRSN] 내부 중

In [30]:
# 2. 복합 조건(no + keyword) 기반 최종 마스터 통합
if all_domain_dfs:
    # 하나의 대형 파일로 단순 통합
    df_total_combined = pd.concat(all_domain_dfs, ignore_index=True)
    total_before = len(df_total_combined)
    print(f"1차 그룹들 결합 직후 단순 합산 크기: {total_before}행")
    
    # data_manage_no와 data_manage_keyword가 "둘 다" 같아야만 중복으로 인정하여 제거
    duplicate_keys = ["data_manage_no", "data_manage_keyword"]
    
    if all(k in df_total_combined.columns for k in duplicate_keys):
        # 복합키 기준 중복 제거
        df_final_master = df_total_combined.drop_duplicates(subset=duplicate_keys, keep="first")
        total_after = len(df_final_master)
        deleted_total = total_before - total_after
        
        # 정렬 (키워드별, 관리번호별 정렬)
        df_final_master = df_final_master.sort_values(by=duplicate_keys).reset_index(drop=True)
        
        # 최종 전체 마스터 파일 저장
        final_output_name = "kf_area_total_merged.csv"
        final_output_path = os.path.join(PROCEED_DIR, final_output_name)
        df_final_master.to_csv(final_output_path, index=False, encoding="utf-8-sig")
        
        print("\n[최종 통합 및 검증 완료]")
        print(f"  - 최종 마스터 전체 행/열 구조: {df_final_master.shape}")
        print(f"  - 복합 조건으로 걸러진 최종 중복 행 수: 총 {deleted_total}개")
        print(f"  - cltur 데이터 수: {len(df_final_master[df_final_master["data_manage_keyword"] == "cltur"])}행")
        print(f"  - prsn 데이터 수: {len(df_final_master[df_final_master["data_manage_keyword"] == "prsn"])}행")
        print(f"  - 전체 마스터 저장 완료: {final_output_path}")
    else:
        print("[ERROR] 기준이 되는 컬럼들이 누락되어 최종 복합 중복 제거를 진행할 수 없습니다.")
else:
    print("[WARNING] 병합할 데이터프레임이 존재하지 않습니다.")

1차 그룹들 결합 직후 단순 합산 크기: 26876행

[최종 통합 및 검증 완료]
  - 최종 마스터 전체 행/열 구조: (26876, 22)
  - 복합 조건으로 걸러진 최종 중복 행 수: 총 0개
  - cltur 데이터 수: 14491행
  - prsn 데이터 수: 12385행
  - 전체 마스터 저장 완료: data/proceed/kf_area_total_merged.csv
